# Решения: permutation practice

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


In [ ]:
def permutation_test_diff(frame, n_iter=3000, seed=0):
    rng = np.random.default_rng(seed)
    conv = frame['converted'].to_numpy()
    mask_b = frame['variant'].to_numpy() == 'B'
    obs = float(conv[mask_b].mean() - conv[~mask_b].mean())
    sims = np.empty(n_iter)
    for i in range(n_iter):
        perm = rng.permutation(conv)
        sims[i] = perm[mask_b].mean() - perm[~mask_b].mean()
    p = float((np.abs(sims) >= abs(obs)).mean())
    return obs, p


obs, p = permutation_test_diff(df, n_iter=3000, seed=37)
mob = df[df['device'] == 'mobile']
des = df[df['device'] == 'desktop']
obs_mob, p_mob = permutation_test_diff(mob, n_iter=3000, seed=38)
obs_des, p_des = permutation_test_diff(des, n_iter=3000, seed=39)
_, p_500 = permutation_test_diff(df, n_iter=500, seed=40)
_, p_2000 = permutation_test_diff(df, n_iter=2000, seed=40)
_, p_8000 = permutation_test_diff(df, n_iter=8000, seed=40)
INTERP = (
    'p-value отвечает на вопрос совместимости данных с H0, а не на вопрос масштаба эффекта. '
    'Всегда интерпретируем p-value вместе с самой разницей конверсий и бизнес-контекстом.'
)
ads = df[df['traffic_source'] == 'ads']
_, p_ads = permutation_test_diff(ads, n_iter=3000, seed=41)
rng = np.random.default_rng(77)
half_idx = rng.choice(df.index.to_numpy(), size=len(df) // 2, replace=False)
half = df.loc[half_idx]
_, p_half = permutation_test_diff(half, n_iter=3000, seed=42)
rng2 = np.random.default_rng(43)
conv = df['converted'].to_numpy()
mask_b = df['variant'].to_numpy() == 'B'
sim_vals = []
for _ in range(1200):
    perm = rng2.permutation(conv)
    sim_vals.append(float(perm[mask_b].mean() - perm[~mask_b].mean()))
sim_table = pd.DataFrame({'sim_diff': sim_vals})
PHACK_NOTE = (
    'Если запускать тест много раз, отбирать удобные подвыборки и останавливать анализ на удачном моменте, '
    'можно получить ложную значимость даже без реального эффекта.'
)
print(round(p, 5), round(p_mob, 5), round(p_des, 5), round(p_ads, 5), round(p_half, 5))